# Lecture 7 — Classification Foundations & Logistic Regression

**BINF 6210/8210: Machine Learning for Bioinformatics**

We use the scikit-learn Wisconsin Diagnostic Breast Cancer dataset for teaching. **Malignant is recoded as the positive class (1)**. This notebook is educational and not a clinical diagnostic tool.

## Learning objectives
- Build a binary logistic-regression classifier.
- Distinguish probabilities from hard labels.
- Interpret coefficients and odds ratios cautiously.
- Examine the effect of changing the classification threshold.
- Use a `Pipeline` to avoid preprocessing leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
X = data.data.copy()
# Recode so malignant = 1 (positive class), benign = 0
y = (data.target == 0).astype(int)
y.name = "malignant"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=6210
)
print(X.shape, y.value_counts(normalize=True).sort_index())

In [ ]:
X.describe().T[["mean","std","min","max"]].head(10)

## Why not linear regression for a binary outcome?
A linear model can predict values outside [0,1]. Logistic regression instead models the **log-odds** linearly and maps them through the sigmoid.

In [ ]:
z = np.linspace(-8,8,400)
sigmoid = 1/(1+np.exp(-z))
plt.figure(figsize=(7,4))
plt.plot(z, sigmoid)
plt.axhline(0.5, linestyle="--")
plt.axvline(0, linestyle="--")
plt.xlabel("Linear predictor η")
plt.ylabel("p = sigmoid(η)")
plt.title("Logistic sigmoid")
plt.show()

## Fit a leakage-safe pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logit = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=6210))
])
logit.fit(X_train, y_train)

print("Training accuracy:", logit.score(X_train, y_train))
print("Test accuracy:", logit.score(X_test, y_test))

## Probabilities versus labels

In [ ]:
y_prob = logit.predict_proba(X_test)[:,1]
y_pred = logit.predict(X_test)

preview = pd.DataFrame({
    "true_malignant": y_test.to_numpy()[:12],
    "P(malignant)": y_prob[:12],
    "predicted": y_pred[:12]
})
preview

## Standardized coefficients and odds ratios
Because predictors are standardized, each coefficient corresponds to approximately a one-standard-deviation increase in that predictor. Correlation among features means coefficients should not be read as independent biomarker importance.

In [ ]:
coef = pd.Series(logit.named_steps["clf"].coef_[0], index=X.columns)
coef_table = pd.DataFrame({
    "coefficient": coef,
    "odds_ratio_per_1_SD": np.exp(coef)
}).sort_values("coefficient")
pd.concat([coef_table.head(5), coef_table.tail(5)])

## Change the threshold

In [ ]:
for threshold in [0.2,0.5,0.8]:
    pred_t = (y_prob >= threshold).astype(int)
    print(threshold, "predicted malignant:", pred_t.sum(), "accuracy:", (pred_t==y_test.to_numpy()).mean())

### In-class questions
1. Why does lowering the threshold usually increase sensitivity?
2. Why is `predict_proba()` more informative than `predict()` for threshold analysis?
3. Does a large standardized coefficient establish causality? Why not?

## Optional: regularization strength

In [ ]:
for C in [0.01, 0.1, 1, 10, 100]:
    m = Pipeline([("scale", StandardScaler()),
                  ("clf", LogisticRegression(C=C, max_iter=5000, random_state=6210))])
    m.fit(X_train,y_train)
    norm = np.linalg.norm(m.named_steps["clf"].coef_)
    print(f"C={C:6g}  train={m.score(X_train,y_train):.3f} test={m.score(X_test,y_test):.3f} ||beta||={norm:.3f}")

## Take-home message
Logistic regression produces class probabilities through a sigmoid transformation of a linear predictor. Hard classification requires a threshold, and regularization/preprocessing choices are part of the model.